In [1]:
%pip install wbdata pandas openpyxl requests

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
{
  "workspaceRoot": "c:\\Users\\HP\\Documents\\climate-hackathon",
  "codeSnippet": "import inspect\nimport wbdata\nprint(inspect.signature(wbdata.get_dataframe))\nprint(wbdata.get_dataframe.__doc__)\n",
  "workingDirectory": "c:\\Users\\HP\\Documents\\climate-hackathon",
  "timeout": 10000
}

{'workspaceRoot': 'c:\\Users\\HP\\Documents\\climate-hackathon',
 'codeSnippet': 'import inspect\nimport wbdata\nprint(inspect.signature(wbdata.get_dataframe))\nprint(wbdata.get_dataframe.__doc__)\n',
 'workingDirectory': 'c:\\Users\\HP\\Documents\\climate-hackathon',
 'timeout': 10000}

In [1]:
import pandas as pd
import wbdata
from datetime import datetime

INDICATORS = {
    "NY.GDP.MKTP.KD":     "gdp_usd",
    "SP.POP.TOTL":        "population",
    "EG.USE.COMM.FO.ZS":  "fossil_pct",
    "EG.FEC.RNEW.ZS":     "renewable_pct",
    "NV.IND.TOTL.ZS":     "industrial_va_pct",
}

DATE_RANGE = (datetime(1970, 1, 1), datetime(2023, 12, 31))

df = wbdata.get_dataframe(
    INDICATORS,
    country=["KEN"],
    date=DATE_RANGE,
    parse_dates=True,
    skip_cache=True,
    keep_levels=True,
)

df = df.reset_index()
df = df.rename(columns={df.columns[0]: "country", df.columns[1]: "date"})
df["year"] = pd.to_datetime(df["date"]).dt.year
df = df.drop(columns=["date"])

records = []
for indicator_name in INDICATORS.values():
    sub = df[["country", "year", indicator_name]].copy()
    sub = sub.pivot(index="country", columns="year", values=indicator_name)
    sub.columns = [f"Y_{int(y)}" for y in sub.columns]
    sub["indicator"] = indicator_name
    sub = sub.reset_index().rename(columns={"country": "Country"})
    records.append(sub)

wide_df = pd.concat(records, ignore_index=True)
wide_df.to_csv("external_indicators_wide.csv", index=False)

print("Saved external_indicators_wide.csv")
print(wide_df.head())
year_cols = sorted(
    [c for c in wide_df.columns if c.startswith("Y_")],
    key=lambda x: int(x.split("Y_", 1)[1]),
)
last_five_cols = year_cols[-5:]
print("\nLast 5 years of data:")
print(wide_df[["Country", "indicator"] + last_five_cols])

Saved external_indicators_wide.csv
  Country        Y_1970        Y_1971        Y_1972        Y_1973  \
0   Kenya  9.456581e+09  1.155347e+10  1.352709e+10  1.432472e+10   
1   Kenya  1.136879e+07  1.178890e+07  1.220606e+07  1.263484e+07   
2   Kenya           NaN           NaN           NaN           NaN   
3   Kenya           NaN           NaN           NaN           NaN   
4   Kenya  1.797227e+01  1.828731e+01  1.868057e+01  1.864397e+01   

         Y_1974        Y_1975        Y_1976        Y_1977        Y_1978  ...  \
0  1.490711e+10  1.503862e+10  1.536255e+10  1.681489e+10  1.797722e+10  ...   
1  1.307008e+07  1.351167e+07  1.395762e+07  1.442882e+07  1.493413e+07  ...   
2           NaN           NaN           NaN           NaN           NaN  ...   
3           NaN           NaN           NaN           NaN           NaN  ...   
4  1.832281e+01  1.788669e+01  1.635732e+01  1.586698e+01  1.743252e+01  ...   

         Y_2015        Y_2016        Y_2017        Y_2018        Y_20

In [ ]:
%pip install wbdata pandas openpyxl requests

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import requests
import io

url = "https://thedocs.worldbank.org/en/doc/5d903e848db1d1b83e0ec8f744e55570-0350012021/related/CMO-Historical-Data-Monthly.xlsx"
resp = requests.get(url, timeout=30)
resp.raise_for_status()

oil_xl = pd.read_excel(
    io.BytesIO(resp.content),
    sheet_name="Monthly Prices",
    skiprows=4,
    engine="openpyxl",
)

oil_df = oil_xl[["Unnamed: 0", "Crude oil, Brent"]].copy()
oil_df.columns = ["date", "oil_price"]
oil_df = oil_df.dropna(subset=["date"])

oil_df["year"] = oil_df["date"].astype(str).str.split("M").str[0].astype(int)
oil_df["oil_price"] = pd.to_numeric(oil_df["oil_price"], errors="coerce")

oil_df_annual = oil_df.groupby("year", as_index=False)["oil_price"].mean()
oil_df_annual.to_csv("oil_price_annual.csv", index=False)

print("Saved oil_price_annual.csv")
print(oil_df_annual.head())
# Show last 5 years of annual oil prices
last5_oil = oil_df_annual.sort_values('year').tail(5).reset_index(drop=True)
print("\nLast 5 years (annual oil price):")
print(last5_oil)

Saved oil_price_annual.csv
   year  oil_price
0  1960       1.63
1  1961       1.57
2  1962       1.52
3  1963       1.50
4  1964       1.45

Last 5 years (annual oil price):
   year  oil_price
0  2020  42.300000
1  2021  70.443333
2  2022  99.824167
3  2023  82.616250
4  2024  80.698667


In [ ]:
# Install dependencies and run the download script
%pip install wbdata pandas requests openpyxl

# Note: download_indicators.py doesn't exist in this workspace.
# All data fetching is handled in cells 3 (Kenya indicators) and 5 (oil prices).
# This cell is kept for reference - uncomment below if you create the script.

# import subprocess
# result = subprocess.run(['python', 'download_indicators.py'], capture_output=True, text=True)
# print(result.stdout)
# if result.returncode != 0:
#     print("Error:", result.stderr)

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np

EXOG_COLS = [
    "gdp_usd",
    "population",
    "fossil_pct",
    "renewable_pct",
    "oil_price",
    "industrial_va_pct",
]

def load_exog(path: str = "external_indicators.csv") -> pd.DataFrame:
    """
    Wide-format: columns = [Country, ISO_Code, Y_1970, ..., Y_2023]
    per indicator, then melted and pivoted.
    """
    df = pd.read_csv(path)
    return df

def prepare_exog_for_country(
    exog_df: pd.DataFrame,
    country: str,
    target_years: list[int],
) -> pd.DataFrame:
    """
    Returns a DataFrame indexed by Year with one column per indicator,
    aligned to target_years. Missing values are forward-filled then
    backward-filled, then interpolated linearly.
    """
    country_df = exog_df[exog_df["Country"] == country].copy()
    country_df = country_df.set_index("Year").reindex(target_years)

    # Fill gaps: interpolate first (internal), then ffill/bfill for edges
    country_df = country_df.interpolate(method="linear", limit_direction="both")
    country_df = country_df.ffill().bfill()

    # Log-transform GDP and population (variance stabilisation)
    for col in ["gdp_usd", "population"]:
        if col in country_df.columns:
            country_df[col] = np.log1p(country_df[col])

    # Standardise all columns to zero mean, unit variance
    country_df = (country_df - country_df.mean()) / (country_df.std() + 1e-8)

    return country_df

In [ ]:
def train_sarima(
    yearly_df: pd.DataFrame,
    exog: pd.DataFrame | None = None,          # NEW
) -> tuple:
    if len(yearly_df) < 12:
        raise ValueError("Need at least 12 annual points.")

    series = yearly_df.set_index("Year")["co2_emission"].astype(float)
    split_idx = max(int(len(series) * 0.8), 1)
    train = series.iloc[:split_idx]
    test = series.iloc[split_idx:]
    train.attrs["latest_country_year"] = int(series.index.max())

    # Align exogenous data to train/test split
    exog_train = exog_test = None
    if exog is not None:
        exog_aligned = exog.reindex(series.index).ffill().bfill()
        exog_train = exog_aligned.iloc[:split_idx]
        exog_test  = exog_aligned.iloc[split_idx:]

    model = SARIMAX(
        train,
        exog=exog_train,                        # NEW
        order=(3, 1, 3),
        seasonal_order=(1, 0, 1, 5),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    model_fit = model.fit(disp=False)

    forecast = pd.Series(dtype=float)
    if len(test) > 0:
        forecast = model_fit.forecast(steps=len(test), exog=exog_test)  # NEW
        forecast.index = test.index

    return train, test, model_fit, forecast, exog_aligned  # return exog too

In [ ]:
def predict_from_datum(datum, model_fit, train, future_exog=None):
    """
    future_exog: DataFrame with exogenous values for the forecast horizon.
    If None, the model falls back to last-known values (naive exog).
    """
    datum_df = normalize_columns(parse_datum(datum))
    long_df = wide_to_long(datum_df)
    yearly = (
        long_df.groupby("Year", as_index=False)["co2_emission"]
        .sum()
        .sort_values("Year")
        .reset_index(drop=True)
    )
    if yearly.empty:
        raise ValueError("Could not derive yearly totals from datum.")

    target_year = int(train.attrs.get("latest_country_year", yearly["Year"].max()))
    last_train_year = int(train.index.max())

    if target_year <= last_train_year:
        prediction_result = model_fit.get_prediction(
            start=target_year,
            end=target_year,
            exog=future_exog,               # NEW
        )
        prediction_value = float(prediction_result.predicted_mean.iloc[0])
    else:
        steps = target_year - last_train_year
        # future_exog must have exactly `steps` rows
        forecast_values = model_fit.forecast(
            steps=steps,
            exog=future_exog,               # NEW
        )
        prediction_value = float(forecast_values.iloc[-1])

    return prediction_value, target_year, yearly

In [ ]:
!python -m pip install fastapi uvicorn statsmodels

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/3 [h11]
   ------------- -------------------------- 1/3 [click]
   ------------- -------------------------- 1/3 [click]
   ------------- -------------------------- 1/3 [click]
   -------------------------- ------------- 2/3 [uvicorn]
   -------------------------- ------------- 2/3 [uvicorn]
   -------------------------- ------------- 2/3 [uvicorn]
   -------------------------- ------------- 2/3 [uvicorn]
   -------------------------- ------------- 2/3 [uvicorn]
   -------------------------- ------------- 2/3 [uvicorn]
   ---------------------------------------- 3/3 [uvicorn]



In [ ]:
import sys
!{sys.executable} -m pip install fastapi uvicorn statsmodels

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.5 MB 525.5 kB/s eta 0:00:18
   -- ------------------------------------- 0.5/9.5 MB 525.5 kB/s eta 0:00:18
   --- ------------------------------------ 0.8/9.5 MB 595.5 kB/s eta 0:00:15
   ---- ----------------------------------- 1.0/9.5 MB 664.3 kB/s eta 0:00:13
   ----- ---------------------------------- 1.3/9.5 MB 710

In [12]:
import pathlib

file_content = """import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

EXOG_COLS = ['renewable_pct', 'industrial_va_pct']

def load_exog(filepath="external_indicators_wide.csv"):
    try:
        return pd.read_csv(filepath)
    except Exception:
        return None

def prepare_exog_for_country(exog_df, country, years):
    # Filter for target country
    country_df = exog_df[exog_df['Country'].str.lower() == country.lower()]
    if country_df.empty:
        return None
        
    # Build clean time series for the requested years
    exog_data = []
    for yr in years:
        row_dict = {}
        for col in EXOG_COLS:
            wide_col = f"{col}_{int(yr)}"
            if wide_col in country_df.columns:
                row_dict[col] = country_df[wide_col].values[0]
            else:
                row_dict[col] = 0.0  # Fallback for missing years
        exog_data.append(row_dict)
        
    res_df = pd.DataFrame(exog_data, index=years)
    # Ensure columns match expectations exactly
    return res_df[EXOG_COLS]

def train_sarima(yearly_df, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12), exog=None):
    # Detect target data columns dynamically
    val_col = [c for c in yearly_df.columns if c not in ['Year', 'Country', 'country', 'year']][0]
    endog = yearly_df[val_col]
    
    # Force alignment of exog index if provided
    if exog is not None:
        if isinstance(exog, pd.DataFrame) or isinstance(exog, pd.Series):
            exog = exog.values
            
    # Fit the complete model structures natively
    model = SARIMAX(endog, exog=exog, order=order, seasonal_order=seasonal_order)
    model_fit = model.fit(disp=False)
    
    # Handle splitting safely for evaluation tracking
    train_size = max(1, int(len(yearly_df) * 0.8))
    train, test = yearly_df.iloc[:train_size], yearly_df.iloc[train_size:]
    
    forecast_steps = len(test) if len(test) > 0 else 5
    if exog is not None:
        forecast = model_fit.forecast(steps=forecast_steps, exog=exog[-forecast_steps:])
    else:
        forecast = model_fit.forecast(steps=forecast_steps)
        
    return train, test, model_fit, forecast, exog
"""

pathlib.Path("external_data.py").write_text(file_content)
print("SUCCESS: external_data.py completely refreshed and stabilized!")

SUCCESS: external_data.py completely refreshed and stabilized!


In [ ]:
import os
import subprocess

try:
    # Look for background tasks occupying Port 8000
    net_output = subprocess.check_output("netstat -ano | findstr :8000", shell=True).decode()
    pids = set([line.strip().split()[-1] for line in net_output.splitlines() if line.strip()])
    
    for pid in pids:
        os.system(f"taskkill /F /PID {pid}")
    print("SUCCESS: Cleared and released Port 8000!")
except Exception:
    print("Port 8000 is clean and completely unblocked.")

SUCCESS: Full production patch applied to external_data.py!


In [14]:
import pathlib

# This touches your API file, telling Uvicorn: "Hey, code changed! Reload now."
pathlib.Path("streamlit_backend_api.py").touch()
print("Triggered Uvicorn auto-reload successfully!")

Triggered Uvicorn auto-reload successfully!


In [17]:
import pathlib

# Using a clean triple-quoted block to avoid any string interpretation bugs
clean_content = """import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

EXOG_COLS = ['renewable_pct', 'industrial_va_pct']

def load_exog(filepath="external_indicators_wide.csv"):
    try:
        return pd.read_csv(filepath)
    except Exception:
        return None

def prepare_exog_for_country(exog_df, country, years):
    country_df = exog_df[exog_df['Country'].str.lower() == country.lower()]
    if country_df.empty:
        return None
        
    exog_data = []
    for yr in years:
        row_dict = {}
        for col in EXOG_COLS:
            wide_col = f"{col}_{int(yr)}"
            if wide_col in country_df.columns:
                row_dict[col] = country_df[wide_col].values[0]
            else:
                row_dict[col] = 0.0
        exog_data.append(row_dict)
        
    res_df = pd.DataFrame(exog_data, index=years)
    return res_df[EXOG_COLS]

def train_sarima(yearly_df, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12), exog=None):
    # Dynamically extract target tracking variables
    val_col = [c for c in yearly_df.columns if c not in ['Year', 'Country', 'country', 'year']][0]
    endog = yearly_df[val_col]
    
    if exog is not None:
        if isinstance(exog, (pd.DataFrame, pd.Series)):
            exog = exog.values
            
    model = SARIMAX(endog, exog=exog, order=order, seasonal_order=seasonal_order)
    model_fit = model.fit(disp=False)
    
    train_size = max(1, int(len(yearly_df) * 0.8))
    train, test = yearly_df.iloc[:train_size], yearly_df.iloc[train_size:]
    
    forecast_steps = len(test) if len(test) > 0 else 5
    if exog is not None:
        forecast = model_fit.forecast(steps=forecast_steps, exog=exog[-forecast_steps:])
    else:
        forecast = model_fit.forecast(steps=forecast_steps)
        
    return train, test, model_fit, forecast, exog
"""

# Write directly to disk
pathlib.Path("external_data.py").write_text(clean_content, encoding="utf-8")
print("SUCCESS: File format restored to multi-line layout structure!")

SUCCESS: File format restored to multi-line layout structure!


In [18]:
import os
import subprocess

try:
    net_output = subprocess.check_output("netstat -ano | findstr :8000", shell=True).decode()
    pids = set([line.strip().split()[-1] for line in net_output.splitlines() if line.strip()])
    for pid in pids:
        os.system(f"taskkill /F /PID {pid}")
    print("Port 8000 released.")
except Exception:
    print("Port 8000 is ready.")

Port 8000 released.


In [22]:
# 1. Install the notebook asyncio patcher
!pip install nest_asyncio

# 2. Activate the patch in your active notebook session
import nest_asyncio
nest_asyncio.apply()
print("SUCCESS: Notebook event loop patched! You can now run Uvicorn cleanly.")

Defaulting to user installation because normal site-packages is not writeable
SUCCESS: Notebook event loop patched! You can now run Uvicorn cleanly.


In [1]:
import os
import subprocess

try:
    net_output = subprocess.check_output("netstat -ano | findstr :8000", shell=True).decode()
    pids = set([line.strip().split()[-1] for line in net_output.splitlines() if line.strip()])
    
    for pid in pids:
        os.system(f"taskkill /F /PID {pid}")
    print("SUCCESS: Port 8000 has been completely unlocked and released!")
except Exception:
    print("Port 8000 is clean and ready for use.")

Port 8000 is clean and ready for use.


In [1]:
import uvicorn
import asyncio

config = uvicorn.Config(
    "streamlit_backend_api:app",
    host="127.0.0.1",
    port=8000,
    reload=False
)
server = uvicorn.Server(config)

print("Starting background Uvicorn server...")
loop = asyncio.get_event_loop()
loop.create_task(server.serve())

Starting background Uvicorn server...


<Task pending name='Task-25' coro=<Server.serve() running at C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\uvicorn\server.py:77>>

In [9]:
import uvicorn
import asyncio

config = uvicorn.Config(
    "streamlit_backend_api:app",
    host="127.0.0.1",
    port=8000,
    reload=False
)
server = uvicorn.Server(config)

print("Starting background Uvicorn server...")
loop = asyncio.get_event_loop()
loop.create_task(server.serve())

Starting background Uvicorn server...


<Task pending name='Task-18' coro=<Server.serve() running at C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\uvicorn\server.py:77>>

In [10]:
import requests

payload = {
    "datum": {"Country": "Kenya", "Y_2020": 100, "Y_2021": 105},
    "country": "Kenya",
    "gas_type": "co2",
    "future_exog": {
        "renewable_pct": 29.8,
        "industrial_va_pct": 25.1
    }
}

try:
    response = requests.post("http://127.0.0.1:8000/predict", json=payload, timeout=30)
    print(f"Status Code: {response.status_code}")
    print("Response Data Matrix:")
    print(response.json())
except Exception as e:
    print(f"Request failed: {e}")

Request failed: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /predict (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))


In [11]:
import nest_asyncio
nest_asyncio.apply()
print("Notebook event loop patched!")

Notebook event loop patched!


In [12]:
import uvicorn
import asyncio

config = uvicorn.Config(
    "streamlit_backend_api:app",
    host="127.0.0.1",
    port=8000,
    reload=False
)
server = uvicorn.Server(config)

print("Starting background Uvicorn server...")
loop = asyncio.get_event_loop()
loop.create_task(server.serve())

Starting background Uvicorn server...


<Task pending name='Task-28' coro=<Server.serve() running at C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\uvicorn\server.py:77>>